# California Single Family Residential - Close Price Prediction

## Project Overview

This notebook develops machine learning models to predict the close price (final sales price) of Single Family Residential (SFR) homes in California using the CRMLS dataset.

## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Model imports
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Try to import XGBoost
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
    print("✓ XGBoost is available")
except ImportError:
    XGBOOST_AVAILABLE = False
    print("⚠ XGBoost not available. Install with: pip install xgboost")

# Set styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print("Libraries imported successfully!")

ModuleNotFoundError: No module named 'matplotlib'

## 2. Load Data

In [ ]:
# Load the data
data_path = 'CRMLSSold202401_filled.csv'  # Update path as needed

df_raw = pd.read_csv(data_path)

print(f"Raw data loaded: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")

# Convert CloseDate to datetime
df_raw['CloseDate'] = pd.to_datetime(df_raw['CloseDate'])

print(f"\nDate range: {df_raw['CloseDate'].min()} to {df_raw['CloseDate'].max()}")
print(f"\nFirst few rows:")
df_raw.head()

In [ ]:
# Check data types and missing values
print("Data Info:")
print(df_raw.info())

print("\n" + "="*80)
print("Property Types:")
print(df_raw['PropertyType'].value_counts())

print("\n" + "="*80)
print("Property SubTypes (top 10):")
print(df_raw['PropertySubType'].value_counts().head(10))

## 3. Data Filtering

### Filter Requirements:
1. PropertyType = 'Residential'
2. PropertySubType = 'SingleFamilyResidence'
3. Remove INVALID transactions (data quality errors):
   - ClosePrice <= 0
   - LivingArea <= 0
   - Missing or invalid Latitude/Longitude
4. **KEEP extreme but valid prices** (no outlier removal)

In [ ]:
df = df_raw.copy()
initial_count = len(df)

print("="*100)
print("DATA FILTERING")
print("="*100)

# Step 1: Filter by PropertyType
df = df[df['PropertyType'] == 'Residential'].copy()
print(f"\n1. PropertyType = 'Residential': {len(df):,} rows (removed {initial_count - len(df):,})")

# Step 2: Filter by PropertySubType
count_before = len(df)
df = df[df['PropertySubType'] == 'SingleFamilyResidence'].copy()
print(f"2. PropertySubType = 'SingleFamilyResidence': {len(df):,} rows (removed {count_before - len(df):,})")

# Step 3: Remove INVALID transactions
print("\n3. Removing INVALID transactions (data quality errors):")

# 3a. ClosePrice <= 0
count_before = len(df)
df = df[df['ClosePrice'] > 0].copy()
print(f"   - ClosePrice <= 0: removed {count_before - len(df):,} rows")

# 3b. LivingArea <= 0
count_before = len(df)
df = df[df['LivingArea'] > 0].copy()
print(f"   - LivingArea <= 0: removed {count_before - len(df):,} rows")

# 3c. Missing Latitude/Longitude
count_before = len(df)
df = df[df['Latitude'].notna() & df['Longitude'].notna()].copy()
print(f"   - Missing Latitude/Longitude: removed {count_before - len(df):,} rows")

# 3d. Invalid coordinates (outside California)
count_before = len(df)
df = df[(df['Latitude'] >= 32.5) & (df['Latitude'] <= 42.0)].copy()
df = df[(df['Longitude'] >= -124.5) & (df['Longitude'] <= -114.0)].copy()
print(f"   - Invalid Latitude/Longitude (outside CA): removed {count_before - len(df):,} rows")

print(f"\n4. IMPORTANT: Keeping extreme but VALID prices (no outlier removal)")
print(f"   - Minimum ClosePrice: ${df['ClosePrice'].min():,.0f}")
print(f"   - Maximum ClosePrice: ${df['ClosePrice'].max():,.0f}")
print(f"   - Mean ClosePrice: ${df['ClosePrice'].mean():,.0f}")
print(f"   - Median ClosePrice: ${df['ClosePrice'].median():,.0f}")

print(f"\nFinal filtered dataset: {len(df):,} rows")
print(f"Total removed: {initial_count - len(df):,} rows ({100*(initial_count - len(df))/initial_count:.1f}%)")

df_filtered = df.copy()

## 4. Exploratory Data Analysis

In [ ]:
# Price distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Regular scale
axes[0].hist(df_filtered['ClosePrice'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Close Price ($)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Close Price', fontsize=14, fontweight='bold')
axes[0].axvline(df_filtered['ClosePrice'].median(), color='red', linestyle='--', label=f"Median: ${df_filtered['ClosePrice'].median():,.0f}")
axes[0].legend()

# Log scale
axes[1].hist(np.log1p(df_filtered['ClosePrice']), bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('log(Close Price)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Distribution of Close Price (Log Scale)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Summary statistics
print("\nPrice Statistics:")
print(df_filtered['ClosePrice'].describe())

In [ ]:
# Geographic distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Scatter plot of properties by location
scatter = axes[0].scatter(df_filtered['Longitude'], df_filtered['Latitude'], 
                         c=df_filtered['ClosePrice'], cmap='viridis', 
                         alpha=0.5, s=10)
axes[0].set_xlabel('Longitude', fontsize=12)
axes[0].set_ylabel('Latitude', fontsize=12)
axes[0].set_title('Geographic Distribution of Properties', fontsize=14, fontweight='bold')
plt.colorbar(scatter, ax=axes[0], label='Close Price ($)')

# Hexbin plot
hexbin = axes[1].hexbin(df_filtered['Longitude'], df_filtered['Latitude'], 
                        C=df_filtered['ClosePrice'], gridsize=30, cmap='YlOrRd', reduce_C_function=np.median)
axes[1].set_xlabel('Longitude', fontsize=12)
axes[1].set_ylabel('Latitude', fontsize=12)
axes[1].set_title('Median Price by Location', fontsize=14, fontweight='bold')
plt.colorbar(hexbin, ax=axes[1], label='Median Close Price ($)')

plt.tight_layout()
plt.show()

## 5. Train/Test Split

- If 6+ months available: Test = last month, Train = 6 months prior
- If single month: Test = last 20%, Train = first 80%

In [ ]:
print("="*100)
print("TRAIN / TEST SPLIT")
print("="*100)

df = df_filtered.copy()

# Find the date range
min_date = df['CloseDate'].min()
max_date = df['CloseDate'].max()
date_span_days = (max_date - min_date).days

print(f"\nDataset date range: {min_date.date()} to {max_date.date()}")
print(f"Total span: {date_span_days} days")

# Check if we have multiple months of data
if date_span_days >= 180:  # At least 6 months
    # Use the standard approach: last month as test
    last_month_start = max_date.replace(day=1)
    print(f"\nUsing STANDARD SPLIT: Last month as test set")
    print(f"Last month starts: {last_month_start.date()}")
    
    # Test set: last month
    df_test = df[df['CloseDate'] >= last_month_start].copy()
    
    # Training set: min 6 months before test month
    train_end = last_month_start - timedelta(days=1)
    train_start = (last_month_start - pd.DateOffset(months=6))
    
    df_train = df[(df['CloseDate'] >= train_start) & 
                  (df['CloseDate'] <= train_end)].copy()
else:
    # Use alternative approach: last 20% as test
    print(f"\nWARNING: Only {date_span_days} days of data available (< 180 days)")
    print(f"Using ALTERNATIVE SPLIT: Last 20% of data as test set")
    
    # Calculate cutoff date (80/20 split)
    cutoff_date = min_date + timedelta(days=int(date_span_days * 0.8))
    
    # Test set: last 20%
    df_test = df[df['CloseDate'] >= cutoff_date].copy()
    
    # Training set: first 80%
    df_train = df[df['CloseDate'] < cutoff_date].copy()

print(f"\nTraining set:")
print(f"  - Date range: {df_train['CloseDate'].min().date()} to {df_train['CloseDate'].max().date()}")
print(f"  - Number of records: {len(df_train):,}")
print(f"  - Price range: ${df_train['ClosePrice'].min():,.0f} to ${df_train['ClosePrice'].max():,.0f}")

print(f"\nTest set:")
print(f"  - Date range: {df_test['CloseDate'].min().date()} to {df_test['CloseDate'].max().date()}")
print(f"  - Number of records: {len(df_test):,}")
print(f"  - Price range: ${df_test['ClosePrice'].min():,.0f} to ${df_test['ClosePrice'].max():,.0f}")

# Create trimmed test set for evaluation (0.5% top and bottom)
print(f"\nCreating trimmed test set for evaluation (removing top and bottom 0.5%):")
lower_percentile = df_test['ClosePrice'].quantile(0.005)
upper_percentile = df_test['ClosePrice'].quantile(0.995)

df_test_eval = df_test[
    (df_test['ClosePrice'] >= lower_percentile) & 
    (df_test['ClosePrice'] <= upper_percentile)
].copy()

print(f"  - Original test set: {len(df_test):,} records")
print(f"  - Trimmed test set: {len(df_test_eval):,} records")
print(f"  - Removed: {len(df_test) - len(df_test_eval)} records")
print(f"  - Price range (trimmed): ${df_test_eval['ClosePrice'].min():,.0f} to ${df_test_eval['ClosePrice'].max():,.0f}")

## 6. Feature Engineering

In [ ]:
def create_features(df):
    """Create features for a dataframe."""
    df_feat = pd.DataFrame()
    
    # Base location features
    df_feat['Latitude'] = df['Latitude']
    df_feat['Longitude'] = df['Longitude']
    
    # Property characteristics
    df_feat['LivingArea'] = df['LivingArea']
    df_feat['BedroomsTotal'] = df['BedroomsTotal'].fillna(df['BedroomsTotal'].median())
    df_feat['BathroomsTotalInteger'] = df['BathroomsTotalInteger'].fillna(df['BathroomsTotalInteger'].median())
    
    # Year built features
    if 'YearBuilt' in df.columns:
        df_feat['YearBuilt'] = df['YearBuilt'].fillna(df['YearBuilt'].median())
        df_feat['PropertyAge'] = 2024 - df_feat['YearBuilt']
        df_feat['PropertyAge_Squared'] = df_feat['PropertyAge'] ** 2
    
    # Lot size
    if 'LotSizeSquareFeet' in df.columns:
        df_feat['LotSizeSquareFeet'] = df['LotSizeSquareFeet'].fillna(df['LotSizeSquareFeet'].median())
        df_feat['LotSize_Log'] = np.log1p(df_feat['LotSizeSquareFeet'])
    
    # Garage and parking
    if 'GarageSpaces' in df.columns:
        df_feat['GarageSpaces'] = df['GarageSpaces'].fillna(0)
    if 'ParkingTotal' in df.columns:
        df_feat['ParkingTotal'] = df['ParkingTotal'].fillna(0)
    
    # Pool indicator
    if 'PoolPrivateYN' in df.columns:
        df_feat['HasPool'] = (df['PoolPrivateYN'] == True).astype(int)
    
    # Fireplace
    if 'FireplacesTotal' in df.columns:
        df_feat['FireplacesTotal'] = df['FireplacesTotal'].fillna(0)
        df_feat['HasFireplace'] = (df_feat['FireplacesTotal'] > 0).astype(int)
    
    # Stories
    if 'Stories' in df.columns:
        stories_map = {'One': 1, 'Two': 2, 'Three': 3, 'ThreeOrMore': 3.5, 
                     'Four': 4, 'FourOrMore': 4.5}
        df_feat['Stories'] = df['Stories'].map(stories_map).fillna(1)
    
    # Location-based engineered features
    df_feat['Lat_Lon_Interaction'] = df_feat['Latitude'] * df_feat['Longitude']
    df_feat['Lat_Squared'] = df_feat['Latitude'] ** 2
    df_feat['Lon_Squared'] = df_feat['Longitude'] ** 2
    
    # Distance to major CA cities
    # Los Angeles: 34.0522° N, -118.2437° W
    # San Francisco: 37.7749° N, -122.4194° W
    # San Diego: 32.7157° N, -117.1611° W
    
    df_feat['Dist_LA'] = np.sqrt(
        (df_feat['Latitude'] - 34.0522)**2 + 
        (df_feat['Longitude'] - (-118.2437))**2
    )
    df_feat['Dist_SF'] = np.sqrt(
        (df_feat['Latitude'] - 37.7749)**2 + 
        (df_feat['Longitude'] - (-122.4194))**2
    )
    df_feat['Dist_SD'] = np.sqrt(
        (df_feat['Latitude'] - 32.7157)**2 + 
        (df_feat['Longitude'] - (-117.1611))**2
    )
    
    # Minimum distance to major city
    df_feat['Min_Dist_Major_City'] = df_feat[['Dist_LA', 'Dist_SF', 'Dist_SD']].min(axis=1)
    
    # Property size features
    df_feat['LivingArea_Log'] = np.log1p(df_feat['LivingArea'])
    df_feat['LivingArea_Squared'] = df_feat['LivingArea'] ** 2
    df_feat['LivingArea_Sqrt'] = np.sqrt(df_feat['LivingArea'])
    
    # Rooms per living area ratio
    df_feat['Bed_per_sqft'] = df_feat['BedroomsTotal'] / (df_feat['LivingArea'] + 1)
    df_feat['Bath_per_sqft'] = df_feat['BathroomsTotalInteger'] / (df_feat['LivingArea'] + 1)
    df_feat['Total_Rooms'] = df_feat['BedroomsTotal'] + df_feat['BathroomsTotalInteger']
    
    # Coastal proximity (longitude-based for CA)
    df_feat['Coastal_Proximity'] = np.abs(df_feat['Longitude'] + 120)
    
    # Latitude zones (Northern vs Southern CA)
    df_feat['Is_Northern_CA'] = (df_feat['Latitude'] > 37).astype(int)
    df_feat['Is_Southern_CA'] = (df_feat['Latitude'] < 35).astype(int)
    df_feat['Is_Central_CA'] = ((df_feat['Latitude'] >= 35) & (df_feat['Latitude'] <= 37)).astype(int)
    
    return df_feat

print("="*100)
print("FEATURE ENGINEERING")
print("="*100)

# Create features for train and test sets
print("\nCreating features for training set...")
X_train = create_features(df_train)
y_train = df_train['ClosePrice'].values

print("Creating features for test set...")
X_test = create_features(df_test)
y_test = df_test['ClosePrice'].values

# Also create for evaluation set
X_test_eval = create_features(df_test_eval)
y_test_eval = df_test_eval['ClosePrice'].values

print(f"\nFeature set created:")
print(f"  - Number of features: {X_train.shape[1]}")
print(f"  - Training samples: {X_train.shape[0]:,}")
print(f"  - Test samples: {X_test.shape[0]:,}")
print(f"  - Test samples (eval): {X_test_eval.shape[0]:,}")

print(f"\nFeatures created:")
for i, col in enumerate(X_train.columns, 1):
    print(f"  {i:2d}. {col}")

# Handle any remaining NaN values
X_train = X_train.fillna(X_train.median())
X_test = X_test.fillna(X_train.median())  # Use train median
X_test_eval = X_test_eval.fillna(X_train.median())

## 7. Feature Scaling

In [ ]:
print("="*100)
print("FEATURE SCALING")
print("="*100)

# Fit scaler on training data only
scaler = StandardScaler()
scaler.fit(X_train)

# Transform all sets
X_train_scaled = pd.DataFrame(
    scaler.transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)
X_test_eval_scaled = pd.DataFrame(
    scaler.transform(X_test_eval),
    columns=X_test_eval.columns,
    index=X_test_eval.index
)

print("Features scaled using StandardScaler (fitted on training data)")
print(f"  - Mean: ~0, Std: ~1")

## 8. Train Models

Training 6 different regression models:
1. Linear Regression
2. Ridge Regression
3. Lasso Regression
4. ElasticNet
5. Random Forest
6. Gradient Boosting

In [2]:
print("="*100)
print("MODEL TRAINING")
print("="*100)

models = {}
predictions = {}

# 1. Linear Regression
print("\n1. Training Linear Regression...")
models['Linear Regression'] = LinearRegression()
models['Linear Regression'].fit(X_train_scaled, y_train)
predictions['Linear Regression'] = {
    'test': models['Linear Regression'].predict(X_test_scaled),
    'eval': models['Linear Regression'].predict(X_test_eval_scaled)
}
print("   :Linear Regression trained")

# 2. Ridge Regression
print("\n2. Training Ridge Regression...")
models['Ridge'] = Ridge(alpha=100.0)
models['Ridge'].fit(X_train_scaled, y_train)
predictions['Ridge'] = {
    'test': models['Ridge'].predict(X_test_scaled),
    'eval': models['Ridge'].predict(X_test_eval_scaled)
}
print("   :Ridge trained")

# 3. Lasso Regression
print("\n3. Training Lasso Regression...")
models['Lasso'] = Lasso(alpha=1000.0, max_iter=5000)
models['Lasso'].fit(X_train_scaled, y_train)
predictions['Lasso'] = {
    'test': models['Lasso'].predict(X_test_scaled),
    'eval': models['Lasso'].predict(X_test_eval_scaled)
}
print("   :Lasso trained")


MODEL TRAINING

1. Training Linear Regression...


NameError: name 'LinearRegression' is not defined